## Analysis Notebook Start

Section Content
- Package imports
- Data imports
- Data cleaning & normalization
- Set common variables

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import datetime
import plotly.graph_objects as go


load_dotenv()

# Common Variables
today = datetime.date.today()
cutoff20 = today.year - 20
cutoff10 = today.year - 10
cutoff5 = today.year - 5
cutoff3 = today.year - 3

ticker = ''
audit = False

path_stockdata = os.path.join(os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList'), ticker)

# Audit Export
path_analysis_csv = os.path.join(path_stockdata, f'{ticker.upper()}--Analysis=DivBond-s1v1.csv')

# Data Import
dp_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--DivPrice_History-s1v1.csv'), index_col=0)
dp_aggr_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Aggregate_Cy_DivPrice_History-s1v1.csv'), index_col=0)
ann10k_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Annual_10K-s1v1.csv'), index_col=0)


In [ ]:
%%capture
# Data Cleaning Div Price
dp_df0['Date'] = pd.to_datetime(dp_df0['Date'])
dp_df1 = dp_df0[dp_df0['Date'].dt.year >= cutoff20]
dp_df1['FwdDiv%'] = dp_df0['FwdDivYield']


# Data Cleaning Aggregate Div Price
dp_aggr_df1 = dp_aggr_df0.query('DateCy >= @cutoff20 and DateCy < @today.year')

# Data Cleaning Annual 10K Data
ann10k_df0 = ann10k_df0.dropna(subset=['FiscalYear'])
ann10k_df0[['FiscalYear', 'FiscalMonth']] = ann10k_df0[['FiscalYear', 'FiscalMonth']].astype(int)
ann10k_df1 = ann10k_df0[ann10k_df0['FiscalYear'] >= cutoff20]

# Margins
margin_df = ann10k_df1[['FiscalYear', 'Revenue', 'GrossProfit', 'OperatingIncome', 'NetIncome', 'OpCash', 'FreeCash', 'CAPEX', 'FFO']]
margin_df['GPM'] = margin_df['GrossProfit'] / margin_df['Revenue']
margin_df['OPM'] = margin_df['OperatingIncome'] / margin_df['Revenue']
margin_df['NPM'] = margin_df['NetIncome'] / margin_df['Revenue']
margin_df['OCM'] = margin_df['OpCash'] / margin_df['Revenue']
margin_df['FCM'] = margin_df['FreeCash'] / margin_df['Revenue']
margin_df['FFOM'] = margin_df['FFO'] / margin_df['Revenue']
margin_df['CXM'] = margin_df['CAPEX'] / margin_df['Revenue']

# DvD
dvd_df = ann10k_df1[['FiscalYear', 'Revenue', 'OpCash', 'FreeCash', 'CAPEX', 'DivCash', 'StockIssue', 'StockBuyBack',
                     'C&E', 'TreasuryStock',  'CurrentAssets', 'TotalAssets', 'ShortDebt', 'LongDebt', 'CurrentLiabilities', 'TotalLiabilities', 'FFO']]

# GRO
gro_df0 = ann10k_df0[['FiscalYear', 'Revenue', 'OpCash', 'DivCash', 'RevPS', 'OpCashPS', 'DivPS']]
gro_df0['RevGro'] = round(gro_df0['Revenue'].pct_change() , 2)
gro_df0['OpCashGro'] = round(gro_df0['Revenue'].pct_change() , 2)
gro_df0['DivGro'] = round(gro_df0['DivCash'].pct_change() , 2)
gro_df0['RevGroPS'] = round(gro_df0['RevPS'].pct_change() , 2)
gro_df0['OpCashGroPS'] = round(gro_df0['OpCashPS'].pct_change() , 2)
gro_df0['DivGroPS'] = round(gro_df0['DivPS'].pct_change() , 2)
gro_df1 = gro_df0.tail(20)


In [ ]:
#dp_df1

In [ ]:
#dp_aggr_df1

In [ ]:
#ann10k_df1

In [ ]:
#margin_df

In [ ]:
#dvd_df

In [ ]:
#gro_df1

## Dividend Value Theory
Steps
1. View aggregate dividend yields for last 10 years
2. View Mean & Medians (M&Ms) for last 20 years
3. Make initial gate prediction
4. Analysis of DYT chart with gates
5. Confirm gate values

In [ ]:
dp_aggr_df1[['DivYieldMin', 'DivYieldMax', 'DivYieldMean', 'DivYieldMedian']].style.format({'DivYieldMin': '{:.2%}', 'DivYieldMax': '{:.2%}',
                                                                                           'DivYieldMean': '{:.2%}', 'DivYieldMedian': '{:.2%}'})


In [ ]:
# Calculate M&M's

dy_mean20 = dp_df1['FwdDiv%'].mean().tolist()
dy_median20 = dp_df1['FwdDiv%'].median().tolist()

dp_l10_df = dp_df1[dp_df1['Date'].dt.year >= cutoff10]
dy_mean10 = dp_l10_df['FwdDiv%'].mean().tolist()
dy_median10 = dp_l10_df['FwdDiv%'].median().tolist()

dp_l5_df = dp_df1[dp_df1['Date'].dt.year >= cutoff5]
dy_mean5 = dp_l5_df['FwdDiv%'].mean().tolist()
dy_median5 = dp_l5_df['FwdDiv%'].median().tolist()

dp_l3_df = dp_df1[dp_df1['Date'].dt.year >= cutoff3]
dy_mean3 = dp_l3_df['FwdDiv%'].mean().tolist()
dy_median3 = dp_l3_df['FwdDiv%'].median().tolist()

print(f'Mean20Yr Div: {round(dy_mean20 * 100, 2)}%')
print(f'Median20Yr Div: {round(dy_median20 * 100, 2)}%')
print()
print(f'Mean10Yr Div: {round(dy_mean10 * 100, 2)}%')
print(f'Median10Yr Div: {round(dy_median10 * 100, 2)}%')
print()
print(f'Mean5Yr Div: {round(dy_mean5 * 100, 2)}%')
print(f'Median5Yr Div: {round(dy_median5 * 100, 2)}%')
print()
print(f'Mean3Yr Div: {round(dy_mean3 * 100, 2)}%')
print(f'Median3Yr Div: {round(dy_median3 * 100, 2)}%')
print()

In [ ]:
gate_dy = 0.00
gate_dy10 = round((gate_dy * .1) + gate_dy, 4)
gate_dy20 = round((gate_dy * .20) + gate_dy, 4)

print(f'The DivBond Gate is: {round(gate_dy * 100, 2)}%')
print(f'The DivBond Gate10 is: {round(gate_dy10 * 100, 2)}%')
print(f'The DivBond Gate20 is: {round(gate_dy20 * 100, 2)}%')

In [ ]:
dyt_fig = go.Figure([
    go.Scatter(name='DYT', y=round(dp_df1['FwdDivYield'] * 100, 2), x=dp_df1['Date'], mode='lines', marker_color='Blue')
])
dyt_fig.add_hline(y=round(gate_dy * 100, 2), line_dash="dash", line_color="red", annotation_text="Gate", annotation_position="right")
dyt_fig.add_hline(y=round(gate_dy10 * 100, 2), line_dash="dash", line_color="yellow", annotation_text="Gate10", annotation_position="right")
dyt_fig.add_hline(y=round(gate_dy20 * 100, 2), line_dash="dash", line_color="green", annotation_text="Gate20", annotation_position="right")
dyt_fig.update_layout(yaxis_title='FwdDivYield %', xaxis_title='FiscalYear', title='DYT', template='plotly_dark')
dyt_fig.show()

## Margins
Steps
1. Profit Margins
2. Cash Margins
3. Dividend Margins

In [ ]:
gpm_mean20 = margin_df['GPM'].tail(20).mean().tolist()
gpm_mean10 = margin_df['GPM'].tail(10).mean().tolist()
gpm_mean5 = margin_df['GPM'].tail(5).mean().tolist()
gpm_mean3 = margin_df['GPM'].tail(3).mean().tolist()


opm_mean20 = margin_df['OPM'].tail(20).mean().tolist()
opm_mean10 = margin_df['OPM'].tail(10).mean().tolist()
opm_mean5 = margin_df['OPM'].tail(5).mean().tolist()
opm_mean3 = margin_df['OPM'].tail(3).mean().tolist()


npm_mean20 = margin_df['NPM'].tail(20).mean().tolist()
npm_mean10 = margin_df['NPM'].tail(10).mean().tolist()
npm_mean5 = margin_df['NPM'].tail(5).mean().tolist()
npm_mean3 = margin_df['NPM'].tail(3).mean().tolist()



In [ ]:
profit_fig1 = go.Figure(data=[
    go.Bar(name='GPM', x=margin_df['FiscalYear'], y=round(margin_df['GPM'] * 100, 2), offsetgroup=1, marker_color='DarkBlue'),
    go.Bar(name='OPM', x=margin_df['FiscalYear'], y=round(margin_df['OPM'] * 100, 2), offsetgroup=2, marker_color='Blue'),
    go.Bar(name='NPM', x=margin_df['FiscalYear'], y=round(margin_df['NPM'] * 100, 2), offsetgroup=3, marker_color='LightBlue'),

])
profit_fig1.update_xaxes(dtick=1)
profit_fig1.update_layout(yaxis_title='Margin %', xaxis_title='FiscalYear', title='Profit Margins', template='plotly_dark')
profit_fig1.show()

In [ ]:

print(f'Mean 20Year GPM: {round(gpm_mean20 * 100, 2)}%')
print(f'Mean 10Year GPM: {round(gpm_mean10 * 100, 2)}%')
print(f'Mean 5Year GPM: {round(gpm_mean5 * 100, 2)}%')
print(f'Mean 3Year GPM: {round(gpm_mean3 * 100, 2)}%')
print()

print(f'Mean 20Year OPM: {round(opm_mean20 * 100, 2)}%')
print(f'Mean 10Year OPM: {round(opm_mean10 * 100, 2)}%')
print(f'Mean 5Year OPM: {round(opm_mean5 * 100, 2)}%')
print(f'Mean 3Year OPM: {round(opm_mean3 * 100, 2)}%')
print()

print(f'Mean 20Year NPM: {round(npm_mean20 * 100, 2)}%')
print(f'Mean 10Year NPM: {round(npm_mean10 * 100, 2)}%')
print(f'Mean 5Year NPM: {round(npm_mean5 * 100, 2)}%')
print(f'Mean 3Year NPM: {round(npm_mean3 * 100, 2)}%')


In [ ]:
ocm_mean20 = margin_df['OCM'].tail(20).mean().tolist()
ocm_mean10 = margin_df['OCM'].tail(10).mean().tolist()
ocm_mean5 = margin_df['OCM'].tail(5).mean().tolist()
ocm_mean3 = margin_df['OCM'].tail(3).mean().tolist()

fcm_mean20 = margin_df['FCM'].tail(20).mean().tolist()
fcm_mean10 = margin_df['FCM'].tail(10).mean().tolist()
fcm_mean5 = margin_df['FCM'].tail(5).mean().tolist()
fcm_mean3 = margin_df['FCM'].tail(3).mean().tolist()

ffom_mean20 = margin_df['FFOM'].tail(20).mean().tolist()
ffom_mean10 = margin_df['FFOM'].tail(10).mean().tolist()
ffom_mean5 = margin_df['FFOM'].tail(5).mean().tolist()
ffom_mean3 = margin_df['FFOM'].tail(3).mean().tolist()

cap_mean20 = margin_df['CXM'].tail(20).mean()
cap_mean10 = margin_df['CXM'].tail(10).mean()
cap_mean5 = margin_df['CXM'].tail(5).mean()
cap_mean3 = margin_df['CXM'].tail(3).mean()

In [ ]:
cash_fig1 = go.Figure(data=[
    go.Bar(name='OCM', x=margin_df['FiscalYear'].tail(20), y=round(margin_df['OCM'].tail(20) * 100, 2), offsetgroup=1, marker_color='DarkBlue'),
    go.Bar(name='FCM', x=margin_df['FiscalYear'].tail(20), y=round(margin_df['FCM'].tail(20) * 100, 2), offsetgroup=2, marker_color='LightBlue'),
    go.Bar(name='FFOM', x=margin_df['FiscalYear'].tail(20), y=round(margin_df['FCM'].tail(20) * 100, 2), offsetgroup=3, marker_color='Blue'),
    go.Bar(name='CXM', x=margin_df['FiscalYear'].tail(20), y=round(margin_df['CXM'].tail(20) * 100, 2).abs(), offsetgroup=4, marker_color='Red'),

])
cash_fig1.update_xaxes(dtick=1)
cash_fig1.update_layout(yaxis_title='Margin %', xaxis_title='FiscalYear', title='Cash Margins', template='plotly_dark')
cash_fig1.show()

In [ ]:
print(f'Mean 20Year OCM: {round(ocm_mean20 * 100, 2)}%')
print(f'Mean 10Year OCM: {round(ocm_mean10 * 100, 2)}%')
print(f'Mean 5Year OCM: {round(ocm_mean5 * 100, 2)}%')
print(f'Mean 3Year OCM: {round(ocm_mean3 * 100, 2)}%')
print()
print(f'Mean 20Year FCM: {round(fcm_mean20 * 100, 2)}%')
print(f'Mean 10Year FCM: {round(fcm_mean10 * 100, 2)}%')
print(f'Mean 5Year FCM: {round(fcm_mean5 * 100, 2)}%')
print(f'Mean 3Year FCM: {round(fcm_mean3 * 100, 2)}%')
print()
print(f'Mean 20Year FFOM: {round(ffom_mean20 * 100, 2)}%')
print(f'Mean 10Year FFOM: {round(ffom_mean10 * 100, 2)}%')
print(f'Mean 5Year FFOM: {round(ffom_mean5 * 100, 2)}%')
print(f'Mean 3Year FFOM: {round(fcm_mean3 * 100, 2)}%')
print()
print(f'Mean 20Year CXM: {round(cap_mean20 * 100, 2)}%')
print(f'Mean 10Year CXM: {round(cap_mean10 * 100, 2)}%')
print(f'Mean 5Year CXM: {round(cap_mean5 * 100, 2)}%')
print(f'Mean 3Year CXM: {round(cap_mean3 * 100, 2)}%')

## DVD - Dividend Vs Debt
Steps
1. Dividend Covered by OpCash
2. Dividend Covered by FreeCash
3. Total Cash On Hand vs Total Debts & CAPEX
4. Operating Cash vs Current Liabilities & CAPEX

In [ ]:
dop_mean20 = (dvd_df['DivCash']/dvd_df['OpCash']).tail(20).mean().tolist()
dop_mean10 = (dvd_df['DivCash']/dvd_df['OpCash']).tail(10).mean().tolist()
dop_mean5 = (dvd_df['DivCash']/dvd_df['OpCash']).tail(5).mean().tolist()
dop_mean3 = (dvd_df['DivCash']/dvd_df['OpCash']).tail(3).mean().tolist()

dfr_mean20 = (dvd_df['DivCash']/dvd_df['FreeCash']).tail(20).mean().tolist()
dfr_mean10 = (dvd_df['DivCash']/dvd_df['FreeCash']).tail(10).mean().tolist()
dfr_mean5 = (dvd_df['DivCash']/dvd_df['FreeCash']).tail(5).mean().tolist()
dfr_mean3 = (dvd_df['DivCash']/dvd_df['FreeCash']).tail(3).mean().tolist()

dffo_mean20 = (dvd_df['DivCash']/dvd_df['FFO']).tail(20).mean().tolist()
dffo_mean10 = (dvd_df['DivCash']/dvd_df['FFO']).tail(10).mean().tolist()
dffo_mean5 = (dvd_df['DivCash']/dvd_df['FFO']).tail(5).mean().tolist()
dffo_mean3 = (dvd_df['DivCash']/dvd_df['FFO']).tail(3).mean().tolist()

In [ ]:
div_fig1 = go.Figure(data=[
    go.Bar(name='DOCM', x=dvd_df['FiscalYear'], y=round((dvd_df['DivCash']/dvd_df['OpCash']).abs() * 100, 2), offsetgroup=1, marker_color='DarkBlue'),
    go.Bar(name='DFCM', x=dvd_df['FiscalYear'], y=round((dvd_df['DivCash']/dvd_df['FreeCash']).abs() * 100, 2), offsetgroup=2, marker_color='Blue'),
    go.Bar(name='DFFOM', x=dvd_df['FiscalYear'], y=round((dvd_df['DivCash']/dvd_df['FFO']).abs() * 100, 2), offsetgroup=3, marker_color='LightBlue')
])
div_fig1.update_xaxes(dtick=1)
div_fig1.update_yaxes(range=[0,100])
div_fig1.update_layout(yaxis_title='Margin %', xaxis_title='FiscalYear', title='Div Margins ', template='plotly_dark')
div_fig1.show()

In [ ]:
print(f'Mean 20Year Div Op Cash Margin: {abs(round(dop_mean20 * 100, 2))}%')
print(f'Mean 10Year Div Op Cash Margin: {abs(round(dop_mean10 * 100, 2))}%')
print(f'Mean 5Year Div Op Cash Margin: {abs(round(dop_mean5 * 100, 2))}%')
print(f'Mean 3Year Div Op Cash Margin: {abs(round(dop_mean3 * 100, 2))}%')
print()
print(f'Mean 20Year Div Free Cash Margin: {abs(round(dfr_mean20 * 100, 2))}%')
print(f'Mean 10Year Div Free Cash Margin: {abs(round(dfr_mean10 * 100, 2))}%')
print(f'Mean 5Year Div Free Cash Margin: {abs(round(dfr_mean5 * 100, 2))}%')
print(f'Mean 3Year Div Free Cash Margin: {abs(round(dfr_mean3 * 100, 2))}%')
print()
print(f'Mean 20Year Div FFO Margin: {abs(round(dffo_mean20 * 100, 2))}%')
print(f'Mean 10Year Div FFO Margin: {abs(round(dffo_mean10 * 100, 2))}%')
print(f'Mean 5Year Div FFO Margin: {abs(round(dffo_mean5 * 100, 2))}%')
print(f'Mean 3Year Div FFO Margin: {abs(round(dffo_mean3 * 100, 2))}%')

In [ ]:
gate_div_oc = 0.00
gate_div_fc = 0.00
gate_div_ffo = 0.00

In [ ]:
div_fig2 = go.Figure(data=[
    go.Bar(name='OpCash', x=dvd_df['FiscalYear'], y=dvd_df['OpCash'], offsetgroup=1, marker_color='DarkBlue'),
    go.Bar(name='FreeCash', x=dvd_df['FiscalYear'], y=dvd_df['FreeCash'], offsetgroup=2, marker_color='Blue'),
    go.Bar(name='Div', x=dvd_df['FiscalYear'], y=dvd_df['DivCash'].abs(), offsetgroup=3, marker_color='DarkGreen'),
    go.Bar(name='BuyBack', x=dvd_df['FiscalYear'], y=(dvd_df['StockIssue'] + dvd_df['StockBuyBack']).abs(), offsetgroup=3, marker_color='Yellow',
            base=dvd_df['DivCash'].abs())
])
div_fig2.update_layout(barmode='group')
div_fig2.update_xaxes(dtick=1)
div_fig2.update_layout(yaxis_title='Million', xaxis_title='FiscalYear', title='Div Coverage ', template='plotly_dark')
div_fig2.show()

In [ ]:
dvd_div10 = dvd_df['DivCash'].tail(10).sum().tolist()
dvd_bb10 = abs(dvd_df['StockBuyBack'] + dvd_df['StockIssue']).tail(10).sum().tolist()
dvd_opcash10 = dvd_df['OpCash'].tail(10).sum().tolist()

print(f'Year10 Dividend Return: $ {abs(dvd_div10)}')
print(f'Year10 Stock Buy Back Return: $ {abs(dvd_bb10)}')
print(f'Year10 % of Op Cash Return Via Div: {abs(round((dvd_div10 / dvd_opcash10) * 100, 2))}%')
print(f'Year10 % of Op Cash Return Via BB: {abs(round((dvd_bb10 / dvd_opcash10) * 100, 2))}%')

In [ ]:
debt_fig1 = go.Figure(data=[
    go.Bar(name='Cash', x=dvd_df['FiscalYear'], y=abs(dvd_df['C&E']), offsetgroup=2, marker_color='DarkBlue'),
    go.Bar(name='OpCash', x=dvd_df['FiscalYear'], y=dvd_df['OpCash'], offsetgroup=2, marker_color='Blue', base=dvd_df['C&E']),
    go.Bar(name='CurrentDebt', x=dvd_df['FiscalYear'], y=(dvd_df['CurrentLiabilities']), offsetgroup=3, marker_color='DarkRed'),
    go.Bar(name='TotalDebt', x=dvd_df['FiscalYear'], y=(dvd_df['TotalLiabilities'] - dvd_df['CurrentLiabilities']), offsetgroup=3,
           marker_color='Red', base=dvd_df['CurrentLiabilities']),
    go.Bar(name='Capex', x=dvd_df['FiscalYear'], y=(abs(dvd_df['CAPEX'])), offsetgroup=4, marker_color='Yellow'),
])
debt_fig1.update_layout(barmode='group')
debt_fig1.update_xaxes(dtick=1)
debt_fig1.update_layout(yaxis_title='Million', xaxis_title='FiscalYear', title='Debt Coverage By Cash', template='plotly_dark')
debt_fig1.show()

In [ ]:
debt_fig2 = go.Figure(data=[
    go.Bar(name='TreasuryStock', x=dvd_df['FiscalYear'], y=abs(dvd_df['TreasuryStock']), offsetgroup=1, marker_color='Green'),
    go.Bar(name='Cash', x=dvd_df['FiscalYear'], y=abs(dvd_df['C&E']), offsetgroup=2, marker_color='DarkBlue'),
    go.Bar(name='OpCash', x=dvd_df['FiscalYear'], y=dvd_df['OpCash'], offsetgroup=2, marker_color='Blue', base=dvd_df['C&E']),
    go.Bar(name='CurrentLiabilities', x=dvd_df['FiscalYear'], y=(dvd_df['CurrentLiabilities']), offsetgroup=3, marker_color='DarkRed'),
    go.Bar(name='TotalDebt', x=dvd_df['FiscalYear'], y=(dvd_df['TotalLiabilities'] - dvd_df['CurrentLiabilities']), offsetgroup=3,
           marker_color='Red', base=dvd_df['CurrentLiabilities']),
    go.Bar(name='Capex', x=dvd_df['FiscalYear'], y=(abs(dvd_df['CAPEX'])), offsetgroup=4, marker_color='Yellow'),
])
debt_fig2.update_layout(barmode='group')
debt_fig2.update_xaxes(dtick=1)
debt_fig2.update_layout(yaxis_title='Million', xaxis_title='FiscalYear', title='Debt Coverage By Cash & Treasury', template='plotly_dark')
debt_fig2.show()

In [ ]:
debt_fig3 = go.Figure(data=[
    go.Bar(name='OpCash', x=dvd_df['FiscalYear'], y=dvd_df['OpCash'], offsetgroup=1, marker_color='Green'),
    go.Bar(name='CurrentLiabilities', x=dvd_df['FiscalYear'], y=(dvd_df['CurrentLiabilities']), offsetgroup=2, marker_color='Red'),
    go.Bar(name='Capex', x=dvd_df['FiscalYear'], y=(abs(dvd_df['CAPEX'])), offsetgroup=3, marker_color='Yellow'),
])
debt_fig3.update_layout(barmode='group')
debt_fig3.update_xaxes(dtick=1)
debt_fig3.update_layout(yaxis_title='Million', xaxis_title='FiscalYear', title='Current Liabilities Coverage', template='plotly_dark')
debt_fig3.show()

## Growth
- Revenue Growth
- Operating Cash Growth
- Dividend Growth

In [ ]:
gro_df1[['FiscalYear', 'Revenue', 'RevGro','RevPS', 'RevGroPS']]

In [ ]:
rev_bottom_3 = gro_df1[['Revenue']].head(3).mean().tolist()
rev_top_3 = gro_df1[['Revenue']].tail(3).mean().tolist()
cagr_rev = ((rev_top_3[0]/rev_bottom_3[0]) ** (1/20)) - 1


rps_bottom_3 = gro_df1[['RevPS']].head(3).mean().tolist()
rps_top_3 = gro_df1[['RevPS']].tail(3).mean().tolist()
cagr_rps = ((rps_top_3[0]/rps_bottom_3[0]) ** (1/20)) - 1

print(f'The CAGR of Total Company Revenue is: {round(cagr_rev * 100,2 )}%')
print(f'The CAGR of Revenue Per Share is: {round(cagr_rps * 100,2 )}%')

In [ ]:
gro_df1[['FiscalYear', 'OpCash', 'OpCashGro','OpCashPS', 'OpCashGroPS']]

In [ ]:
oc_bottom_3 = gro_df1[['OpCash']].head(3).mean().tolist()
oc_top_3 = gro_df1[['OpCash']].tail(3).mean().tolist()
cagr_opcash = ((oc_top_3[0]/oc_bottom_3[0]) ** (1/20)) - 1


ocps_bottom_3 = gro_df1[['OpCashPS']].head(3).mean().tolist()
ocps_top_3 = gro_df1[['OpCashPS']].tail(3).mean().tolist()
cagr_opcash_ps = ((ocps_top_3[0]/ocps_bottom_3[0]) ** (1/20)) - 1

print(f'The CAGR of Total Company Operating Cash is: {round(cagr_opcash * 100,2 )}%')
print(f'The CAGR of Operating Cash Per Share is: {round(cagr_opcash_ps * 100,2 )}%')

In [ ]:
gro_df1[['FiscalYear', 'DivCash', 'DivGro','DivPS', 'DivGroPS']]

In [ ]:

divgro_mean20 = gro_df1[['DivGro']].tail(20).mean().tolist()
divgro_median20 = gro_df1[['DivGro']].tail(20).median().tolist()
dpsgro_mean20 = gro_df1[['DivGroPS']].tail(20).mean().tolist()
dpsgro_median20 = gro_df1[['DivGroPS']].tail(20).median().tolist()

divgro_mean10 = gro_df1[['DivGro']].tail(10).mean().tolist()
divgro_median10 = gro_df1[['DivGro']].tail(10).median().tolist()
dpsgro_mean10 = gro_df1[['DivGroPS']].tail(10).mean().tolist()
dpsgro_median10 = gro_df1[['DivGroPS']].tail(10).median().tolist()

divgro_mean5 = gro_df1[['DivGro']].tail(5).mean().tolist()
divgro_median5 = gro_df1[['DivGro']].tail(5).median().tolist()
dpsgro_mean5 = gro_df1[['DivGroPS']].tail(5).mean().tolist()
dpsgro_median5 = gro_df1[['DivGroPS']].tail(5).median().tolist()

divgro_mean3 = gro_df1[['DivGro']].tail(3).mean().tolist()
divgro_median3 = gro_df1[['DivGro']].tail(3).median().tolist()
dpsgro_mean3 = gro_df1[['DivGroPS']].tail(3).mean().tolist()
dpsgro_median3 = gro_df1[['DivGroPS']].tail(3).median().tolist()

print(f'Year20 DivGro Mean: {round(divgro_mean20[0] * 100,2 )}% ')
print(f'Year20 DivGro Median: {round(divgro_median20[0] * 100,2 )}% ')
print(f'Year20 DpsGro Mean: {round(dpsgro_mean20[0] * 100,2 )}% ')
print(f'Year20 DpsGro Median: {round(dpsgro_median20[0] * 100,2 )}% ')
print()
print(f'Year10 DivGro Mean: {round(divgro_mean10[0] * 100,2 )}% ')
print(f'Year10 DivGro Median: {round(divgro_median10[0] * 100,2 )}% ')
print(f'Year10 DpsGro Mean: {round(dpsgro_mean10[0] * 100,2 )}% ')
print(f'Year10 DpsGro Median: {round(dpsgro_median10[0] * 100,2 )}% ')
print()
print(f'Year5 DivGro Mean: {round(divgro_mean5[0] * 100,2 )}% ')
print(f'Year5 DivGro Median: {round(divgro_median5[0] * 100,2 )}% ')
print(f'Year5 DpsGro Mean: {round(dpsgro_mean5[0] * 100,2 )}% ')
print(f'Year5 DpsGro Median: {round(dpsgro_median5[0] * 100,2 )}% ')
print()
print(f'Year3 DivGro Mean: {round(divgro_mean3[0] * 100,2 )}% ')
print(f'Year3 DivGro Median: {round(divgro_median3[0] * 100,2 )}% ')
print(f'Year3 DpsGro Mean: {round(dpsgro_mean3[0] * 100,2 )}% ')
print(f'Year3 DpsGro Median: {round(dpsgro_median3[0] * 100,2 )}% ')

## Notebook End Audit Ouput

In [ ]:
audit_json = {
    "analysis_type": "div-bond",
    "analysis_date": today.strftime('%Y-%m-%d'),
    "last_df_date":dp_df1['Date'].iloc[-1].strftime('%Y-%m-%d'),
    "dy_mean20": dy_mean20,
    "dy_median20": dy_median20,
    "dy_mean10": dy_mean10,
    "dy_median10": dy_median10,
    "dy_mean5": dy_mean5,
    "dy_median5": dy_median5,
    "dy_mean3": dy_mean3,
    "dy_median3": dy_median3,
    "divgro_mean20": divgro_mean20[0],
    "divgro_median20": divgro_median20[0],
    "dpsgro_mean20": dpsgro_mean20[0],
    "dpsgro_median20": dpsgro_median20[0],
    "divgro_mean10": divgro_mean10[0],
    "divgro_median10": divgro_median10[0],
    "dpsgro_mean10": dpsgro_mean10[0],
    "dpsgro_median10": dpsgro_median10[0],
    "divgro_mean5": divgro_mean5[0],
    "divgro_median5": divgro_median5[0],
    "dpsgro_mean5": dpsgro_mean5[0],
    "dpsgro_median5": dpsgro_median5[0],
    "divgro_mean3": divgro_mean3[0],
    "divgro_median3": divgro_median3[0],
    "dpsgro_mean3": dpsgro_mean3[0],
    "dpsgro_median3": dpsgro_median3[0],
    "total10_div_return": dvd_div10,
    "total_10_buyback_return": dvd_bb10,
    "total10_opcash": dvd_opcash10,
    "cagr_rev": cagr_rev,
    "cagr_rps": cagr_rps,
    "cagr_opcash": cagr_opcash,
    "cagr_opcash_ps": cagr_opcash_ps,
    "gate_dy": gate_dy,
    "gate_dy10": gate_dy10,
    "gate_dy20": gate_dy20,
    "gate_div_oc": gate_div_oc,
    "gate_div_fc": gate_div_fc,
    "gate_div_ffo": gate_div_ffo
}

audit_json

In [ ]:
audit_df = pd.DataFrame([audit_json])
audit_df

if audit == True:
    if os.path.isfile(path_analysis_csv) :
        audit_df.to_csv(path_analysis_csv, mode='a', header=False, index=False)
    else:
        audit_df.to_csv(path_analysis_csv, mode='w', header=True, index=False)
else:
    print(f'Audit = {audit}')